# Semantic Segmentation + Vision-based Heading Controller Pipeline

End-to-end notebook for the full pipeline:

```
Video frame
    │
    ├─── [optional] cv2.remap (camera undistortion)
    │
    ▼
PIDNet-S segmentation  →  (H, W) label mask  (classes 0-5)
    │
    ▼
┌────────────────────────────────────────────────┐
│  PLUGGABLE HEADING CONTROLLER  (Cell 4 below)  │
│  swap this cell with your own implementation   │
└────────────────────────────────────────────────┘
    │
    ▼
omega_cmd (rad/s) + v_cmd (m/s)
    │
    ▼
controller_output.csv  +  annotated video
```

**Class legend:**  0=background · 1=human · 2=obstacle · 3=road · 4=sidewalk · 5=speed_breaker


In [3]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Make project source importable from the notebook
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path().resolve().parent   # notebooks/ → project root
HEADING_DIR  = PROJECT_ROOT / "src" / "heading_controller"
SEG_DIR      = PROJECT_ROOT / "src" / "segmentation"
PIDNET_DIR   = PROJECT_ROOT / "third_party" / "PIDNet"

for _p in (str(HEADING_DIR), str(SEG_DIR), str(PIDNET_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import cv2
import numpy as np
import torch

print(f"Project root : {PROJECT_ROOT}")
print(f"PIDNet root  : {PIDNET_DIR}  (exists: {PIDNET_DIR.is_dir()})")
print(f"torch        : {torch.__version__}  CUDA: {torch.cuda.is_available()}")


ModuleNotFoundError: No module named 'cv2'

## Cell 2 — Configuration

Edit the paths and flags below to match your setup.  Everything else in the notebook reads from these variables.


In [ ]:
# ── Input ──────────────────────────────────────────────────────────────────
VIDEO_PATH      = str(PROJECT_ROOT / "input_vid.mp4")   # ← change if needed
WEIGHTS_PATH    = str(PROJECT_ROOT / "checkpoints" / "segmentation" / "pidNet_wieghts.pt")
CFG_PATH        = str(PROJECT_ROOT / "src" / "segmentation" / "configs" / "pidnet_robotics_6class.yaml")

# Optional: path to calibration_data.npz  (set to None to skip undistortion)
CALIBRATION_NPZ = None   # e.g. str(PROJECT_ROOT / "calibration_data.npz")

# ── Output ─────────────────────────────────────────────────────────────────
OUTPUT_DIR      = str(PROJECT_ROOT / "results" / "notebook_run")

# ── Performance ────────────────────────────────────────────────────────────
# Run PIDNet every N frames, reusing the previous mask for frames in between.
# 1 = segment every frame  |  3 = ~3× faster, slight accuracy loss on CPU
SEG_EVERY       = 1
FLOW_EVERY      = 1      # same idea for optical flow (1 = every frame)

# ── Debug / dev ────────────────────────────────────────────────────────────
MAX_FRAMES      = 0      # 0 = process all frames; set e.g. 60 for a quick test
SAVE_MASKS      = False  # save per-frame .npy label maps
SAVE_SEG_PNG    = False  # save per-frame coloured segmentation PNGs

print("Configuration set. Run subsequent cells to build the model and execute the pipeline.")


Configuration set. Run subsequent cells to build the model and execute the pipeline.


## Cell 4 — Pluggable Heading Controller

**This is the cell your team member should edit.**

The function `heading_fn(seg_mask, new_K)` is the only thing that needs to change to swap in a different heading algorithm.  The rest of the pipeline (segmentation, optical flow, braking logic, rendering, CSV output) stays identical.

**Interface contract:**
- Input:  `seg_mask` — `(H, W)` int32 label map (classes 0-5)
- Input:  `new_K` — `(3, 3)` float64 calibrated camera matrix, or `None` if no calibration was loaded
- Output: `(omega_cmd, e_psi)` — angular velocity command (rad/s) and heading error (float)

The default implementation below uses the built-in `HeadingController`.  Replace the body of `heading_fn` with your own logic, or set `heading_fn = None` to fall back to the internal controller inside `run_pipeline`.


In [12]:
from heading_controller import HeadingController, DEFAULT_CONFIG

# One shared controller instance (keeps PD state between frames)
_controller = HeadingController(DEFAULT_CONFIG)

# ============================================================
#  DEFAULT implementation — uses built-in PD heading controller
#  Replace the body below with your own heading logic.
# ============================================================
def heading_fn(seg_mask: np.ndarray, new_K) -> tuple:
    """
    Compute angular velocity command from the segmentation mask.

    Parameters
    ----------
    seg_mask : (H, W) int32   semantic label map  (0-5)
    new_K    : (3,3) float64  calibrated camera matrix, or None

    Returns
    -------
    omega_cmd : float   yaw-rate command  (rad/s)
    e_psi     : float   heading error
    """
    if new_K is not None:
        # Calibrated path: normalised heading error  e = (road_cx - cx) / fx
        return _controller.compute_omega_cmd_calibrated(seg_mask, new_K)
    else:
        # Pixel path: error = lane_centre_x - image_centre_x  (pixels)
        return _controller.compute_omega_cmd(seg_mask)

# ============================================================
#  CUSTOM implementation template (comment out the default above
#  and uncomment + edit this block):
# ============================================================
# def heading_fn(seg_mask: np.ndarray, new_K) -> tuple:
#     road_mask = (seg_mask == 3) | (seg_mask == 4)   # road + sidewalk
#     if not road_mask.any():
#         return 0.0, 0.0
#     _, x_idx = np.where(road_mask)
#     road_cx = float(np.mean(x_idx))
#     cx = float(new_K[0, 2]) if new_K is not None else seg_mask.shape[1] / 2.0
#     fx = float(new_K[0, 0]) if new_K is not None else 1.0
#     e = (road_cx - cx) / fx if new_K is not None else (road_cx - cx)
#     K_yaw = 1.5
#     omega_cmd = float(np.clip(K_yaw * e, -1.0, 1.0))
#     return omega_cmd, e

print("heading_fn defined — ready to run the pipeline.")


ModuleNotFoundError: No module named 'heading_controller'

## Cell 6 — Build PIDNet model + precompute camera calibration maps


In [ ]:
import torch.nn.functional as F
from configs import config as pidnet_cfg
from configs import update_config
import models
from pidnet_inference import choose_device, load_pidnet_checkpoint, predict_mask_logits

class _CfgArgs:
    def __init__(self, cfg_file):
        self.cfg  = cfg_file
        self.opts = []

update_config(pidnet_cfg, _CfgArgs(CFG_PATH))

DEVICE      = choose_device()
MODEL_NAME  = pidnet_cfg.MODEL.NAME
NUM_CLASSES = pidnet_cfg.DATASET.NUM_CLASSES
ALIGN_CORS  = bool(getattr(pidnet_cfg.MODEL, "ALIGN_CORNERS", True))
OUT_IDX     = int(pidnet_cfg.TEST.OUTPUT_INDEX)
INPUT_SIZE  = (int(pidnet_cfg.TEST.IMAGE_SIZE[0]), int(pidnet_cfg.TEST.IMAGE_SIZE[1]))

model = models.pidnet.get_pred_model(MODEL_NAME, NUM_CLASSES)
n = load_pidnet_checkpoint(model, Path(WEIGHTS_PATH), DEVICE)
model = model.to(DEVICE)
model.eval()
print(f"Model  : {MODEL_NAME}  classes={NUM_CLASSES}  device={DEVICE}")
print(f"Loaded : {n} tensors from checkpoint")

# ── Camera calibration (optional) ──────────────────────────────────────────
undistort_maps = None
new_K = None

if CALIBRATION_NPZ is not None:
    _cap = cv2.VideoCapture(VIDEO_PATH)
    _w = int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    _h = int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    _cap.release()

    cal   = np.load(CALIBRATION_NPZ)
    K_cam = cal["K"].astype(np.float64)
    dist  = cal["dist"].astype(np.float64)

    new_K, _ = cv2.getOptimalNewCameraMatrix(K_cam, dist, (_w, _h), alpha=1, newImgSize=(_w, _h))
    map1, map2 = cv2.initUndistortRectifyMap(K_cam, dist, None, new_K, (_w, _h), cv2.CV_32FC1)
    undistort_maps = (map1, map2)

    print(f"\nCalibration loaded: {CALIBRATION_NPZ}")
    print(f"new_K  fx={new_K[0,0]:.1f}  fy={new_K[1,1]:.1f}  cx={new_K[0,2]:.1f}  cy={new_K[1,2]:.1f}")
else:
    print("\nNo calibration file set — undistortion skipped.")

## Cell 8 — Run the full pipeline

In [ ]:
from main import run_pipeline

# Reset the controller's PD state before each run
_controller.reset()

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
masks_dir  = out_dir / "masks_npy"
seg_dir    = out_dir / "seg_png"
if SAVE_MASKS:   masks_dir.mkdir(parents=True, exist_ok=True)
if SAVE_SEG_PNG: seg_dir.mkdir(parents=True, exist_ok=True)

_CLASS_COLORS_RGB = np.array([
    [20,24,33],[51,122,183],[64,145,108],[229,126,49],[214,48,49],[243,196,15]
], dtype=np.uint8)

_frame_count = [0]   # mutable counter for mask_fn closure

def _mask_fn(frame_idx: int, frame_bgr: np.ndarray) -> np.ndarray:
    if MAX_FRAMES > 0 and _frame_count[0] >= MAX_FRAMES:
        return np.full(frame_bgr.shape[:2], 3, dtype=np.int32)
    _frame_count[0] += 1
    with torch.inference_mode():
        logits = predict_mask_logits(model, frame_bgr, INPUT_SIZE, DEVICE, ALIGN_CORS, OUT_IDX)
        h, w   = frame_bgr.shape[:2]
        logits = F.interpolate(logits, size=(h, w), mode="bilinear", align_corners=ALIGN_CORS)
        label  = torch.argmax(logits.squeeze(0), dim=0).cpu().numpy().astype(np.uint8)
    if SAVE_MASKS:
        np.save(str(masks_dir / f"frame_{frame_idx:06d}.npy"), label.astype(np.int32))
    if SAVE_SEG_PNG:
        bgr = cv2.cvtColor(_CLASS_COLORS_RGB[np.clip(label, 0, 5)], cv2.COLOR_RGB2BGR)
        cv2.imwrite(str(seg_dir / f"frame_{frame_idx:06d}.png"), bgr)
    return label.astype(np.int32)

run_pipeline(
    video_path     = VIDEO_PATH,
    masks_dir      = None,
    output_csv     = str(out_dir / "controller_output.csv"),
    output_video   = str(out_dir / "controller_output.mp4"),
    visualise      = False,          # always off in notebook
    flow_every     = FLOW_EVERY,
    config         = dict(DEFAULT_CONFIG),
    mask_fn        = _mask_fn,
    undistort_maps = undistort_maps,
    new_K          = new_K,
    heading_fn     = heading_fn,     # ← plug-in from Cell 4
    seg_every      = SEG_EVERY,
)

print(f"\nOutputs saved to: {out_dir}")

## Cell 10 — Results: plot controller output CSV

In [ ]:
import csv
import matplotlib.pyplot as plt

csv_path = out_dir / "controller_output.csv"
rows = list(csv.DictReader(open(csv_path)))

frames    = [int(r["frame"])     for r in rows]
v_cmd     = [float(r["v_cmd"])   for r in rows]
omega_cmd = [float(r["omega_cmd"]) for r in rows]
e_psi     = [float(r["e_psi"])   for r in rows]
proc_ms   = [float(r["proc_ms"]) for r in rows]
d_obs     = [float(r["d_obstacle"])     for r in rows]
d_hum     = [float(r["d_human"])        for r in rows]
d_sb      = [float(r["d_speed_breaker"]) for r in rows]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle("Controller Output", fontsize=14)

axes[0].plot(frames, v_cmd,     label="v_cmd (m/s)",   color="steelblue")
axes[0].plot(frames, omega_cmd, label="ω_cmd (rad/s)", color="darkorange")
axes[0].axhline(0, color="gray", linewidth=0.5)
axes[0].set_ylabel("Commands"); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].plot(frames, e_psi, color="purple")
axes[1].axhline(0, color="gray", linewidth=0.5)
axes[1].set_ylabel("Heading error e_ψ"); axes[1].grid(True, alpha=0.3)

axes[2].plot(frames, d_obs, label="obstacle", color="red")
axes[2].plot(frames, d_hum, label="human",    color="dodgerblue")
axes[2].plot(frames, d_sb,  label="spd_brkr", color="gold")
axes[2].axhline(0.25, color="red",    linewidth=0.8, linestyle="--", label="emergency")
axes[2].axhline(0.50, color="orange", linewidth=0.8, linestyle="--", label="caution")
axes[2].set_ylabel("Proximity score\n(1=safe, 0=close)")
axes[2].legend(fontsize=8, ncol=3); axes[2].grid(True, alpha=0.3)

axes[3].plot(frames, proc_ms, color="gray")
axes[3].set_ylabel("proc_ms"); axes[3].set_xlabel("Frame"); axes[3].grid(True, alpha=0.3)
avg_ms = sum(proc_ms) / len(proc_ms)
axes[3].axhline(avg_ms, color="red", linewidth=0.8, linestyle="--",
                label=f"avg {avg_ms:.1f} ms  ({1000/avg_ms:.2f} FPS)")
axes[3].legend(fontsize=8)

plt.tight_layout()
plt.show()
print(f"Processed {len(rows)} frames  |  avg {avg_ms:.1f} ms/frame  |  {1000/avg_ms:.2f} FPS")